# Unit 1: Differentiable Approximations of Minimum Functions

[Course overview](00_overview.ipynb)

<ul>
<li><a href="#learn" style="color:#E65100;"><strong>Overview and Learning Objectives</strong></a></li>
<li><a href="#softmin" style="color:black;">The Soft Minimum</a></li>
<li><a href="#softmin_1d" style="color:black;">Softmin in One Dimension</a></li>
<li><a href="#softmin_2d" style="color:black;">Softmin in Two Dimensions</a></li>
<li><a href="#alternatives" style="color:black;">Further Minimum Approximations</a></li>
<li><a href="#why_softmin" style="color:#1B5E20;">Background: Why SDTW Uses Softmin</a></li>
<li><a href="#exercise_properties" style="color:#006064;">Exercise 1: Softmin Properties</a></li>
<li><a href="#exercise_stability" style="color:#006064;">Exercise 2: Numerical Stability</a></li>
</ul>

<a id="learn"></a>

<div style="
  background-color: #FFE0B2A0;
  border: 1px solid #FFC671A0;
  padding-top: 10px;
  padding-bottom: 10px;
  padding-left: 20px;
  padding-right: 20px;
  margin-top: 10px;
  margin-bottom: 10px;
">

<h2 style="color:#E65100; margin-top:0;">Overview and Learning Objectives</h2>

<p style="color:#E65100;">First, we study softmin and its gradients in 1D and 2D. Then we compare sparsemin and smoothmin, explain the algebraic
         reason for choosing softmin in SDTW, and consider numerical stability.</p>

</div>

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

from ddtw.backend.backend_torch import _softmin as softmin
from ddtw.backend.backend_torch import _sparsemin as sparsemin
from ddtw.backend.backend_torch import _smoothmin as smoothmin

# --- Figure display and style ---
%config InlineBackend.figure_formats = ['svg']
plt.rcParams.update({"font.size": 8, "axes.titlesize": 9})

<a id="softmin"></a>

## The Soft Minimum


For $a=(x_1,\ldots,x_K)$ and $\gamma>0$, we define the softmin function and its gradient

$$\mu^\gamma(a)=-\gamma\log\sum_{k=1}^K e^{-a_k/\gamma},\qquad
\frac{\partial\mu^\gamma}{\partial a_k}
=\frac{e^{-a_k/\gamma}}{\sum_{j=1}^K a^{-a_j/\gamma}}.$$

The gradient distributes unit mass among the inputs, assigning larger weight to smaller input values.
Small temperatures approximate a hard minimum. A large temperature spreads the
gradient and approaches the arithmetic mean as $\gamma\to\infty$.

<a id="softmin_1d"></a>

## Softmin in One Dimension

We vary $a_1\in[0,4]$ and keep $a_2=2$. The upper plots show the function values; the lower plots show the derivative with respect to $a_1$. For the hard minimum, this derivative switches at $a_1=a_2$. How do the approximations behave near this point?

In [ ]:
min_functions = [("Softmin", softmin)]
gammas = [1e-10, 0.5, 1.0, 2.0]

a1 = np.linspace(0, 4, 100)
a2 = 2

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(4.2, 4), squeeze=False)
for i_func, (name, min_func) in enumerate(min_functions):
    for i_gamma, gamma in enumerate(gammas):
        vals = []
        grads = []
        for a1_ in a1:
            val, grad = min_func(torch.tensor([a1_, a2]), gamma=gamma)
            vals.append(val.item())
            grads.append(grad[0].item())
        ax[0, i_func].plot(a1, vals, label="gamma")
        ax[0, i_func].set_xticks([0, 1, 2, 3, 4])
        ax[0, i_func].set_title(name)

        ax[1, i_func].plot(a1, grads, label="gamma")
        ax[1, i_func].set_xlabel("a1")
        ax[1, i_func].set_xticks([0, 1, 2, 3, 4])
        # val, grad = min_func(torch.tensor([a1, a2]), gamma=gamma, dim=0)
ax[0, 0].set_ylabel("$\mu(a_1, 2)$")
ax[1, 0].set_ylabel("$\partial \mu(a_1, 2)/\partial a_1$")

ax[0, 0].legend(
    [f"$\gamma={gamma}$" for gamma in gammas], loc="lower right", fontsize=8
)

for a in ax.flatten():
    a.grid()
    a.set_xlim([0, 4])

fig.tight_layout()
plt.show()

<a id="softmin_2d"></a>

## Softmin in Two Dimensions

We now vary both $a_1,a_2\in[0,4]$. The surfaces show the function values, and the arrows show both gradient components. Compare slices through these surfaces with the 1D plots above. A small positive $\gamma$ approximates the hard minimum; larger values make the transition less abrupt.

In [ ]:
a1 = np.arange(0, 4.25, 0.25)
a2 = np.arange(0, 4.25, 0.25)
gammas = [1e-10, 0.5, 2.0]

cmap_rbg = LinearSegmentedColormap.from_list(
    "BlueBlackRed", ["tab:blue", "black", "tab:red"]
)

In [ ]:
for name, min_func in min_functions:
    fig, ax = plt.subplots(2, 3, figsize=(6.2, 4.2))

    for i, gamma in enumerate(gammas):
        a1_, a2_ = np.meshgrid(a1, a2)

        U = np.zeros_like(a1_)
        V = np.zeros_like(a2_)

        S = np.zeros_like(a1_)

        for n in range(a1_.shape[0]):
            for m in range(a1_.shape[1]):
                val, grads = min_func(
                    torch.tensor([a1_[n, m], a2_[n, m]]), gamma=gamma
                )

                S[n, m] = val
                U[n, m] = grads[0]
                V[n, m] = grads[1]

        # ax[0,i].contour(a1_, a2_, S, cmap="viridis", levels=20)
        ax[0, i].remove()
        ax[0, i] = fig.add_subplot(2, 3, i + 1, projection="3d")
        ax[0, i].plot_surface(
            a1_,
            a2_,
            S,
            cmap="hot_r",
            edgecolor="black",
            linewidth=0.12,
            antialiased=True,
        )
        ax[0, i].set(
            xlim=[0, 4],
            ylim=[0, 4],
            xticks=[0, 1, 2, 3, 4],
            yticks=[0, 1, 2, 3, 4],
            box_aspect=(1, 1, 0.75),
        )
        ax[0, i].view_init(elev=15, azim=-115)

        ax[0, i].set_xlabel(r"$a_1$", labelpad=-4)
        ax[0, i].set_ylabel(r"$a_2$", labelpad=-4)
        ax[0, i].zaxis.set_rotate_label(False)  # disable automatic rotation
        ax[0, i].set_zlabel(r"$\mu(a_1,a_2)$", rotation=90, labelpad=-4)#, labelpad=2)

        # Move tick labels closer to the grid
        ax[0, i].tick_params(axis="x", pad=-2)
        ax[0, i].tick_params(axis="y", pad=-2)
        ax[0, i].tick_params(axis="z", pad=0)

        ax[1, i].quiver(
            a1, a2, U, V, U - V, cmap=cmap_rbg, headwidth=3, width=0.01
        )
        ax[1, i].set_xlabel("a1")

        ax[1, i].set_xlim([0, 4])
        ax[1, i].set_ylim([0, 4])
        ax[0, i].set_title("gamma = %.1f" % gamma)

    ax[0, 0].set_ylabel("a2")
    ax[1, 0].set_ylabel("a2")
    fig.suptitle(name, fontsize=9)
    fig.tight_layout()
    plt.show()

<div style="
  background-color: #FFE0B2A0;
  border: 1px solid #FFC671A0;
  padding-top: 10px;
  padding-bottom: 10px;
  padding-left: 20px;
  padding-right: 20px;
  margin-top: 10px;
  margin-bottom: 10px;
">

<h2 style="color:#E65100; margin-top:0;">Attention: Values and Gradients</h2>

<p style="color:#E65100;">
          A soft-minimum value can be negative even when all inputs are positive.
          The downward offset does not change the nonnegative, unit-sum gradient.
          At a tie the hard minimum has no unique derivative; the small positive
          temperature used in the plots shares gradient mass across equal inputs.
          </p>

</div>

<a id="alternatives"></a>

## Further Minimum Approximations


Softmin is one possible differentiable approximation of the minimum function. The *d*DTW toolbox also provides
**sparsemin** and **smoothmin**. We repeat the same experiments to compare their
values and gradients on identical inputs and temperatures.

Sparsemin minimizes a quadratically regularized weighted average over the
probability simplex:

$$\mu^\gamma_{\rm sparse}(a)=\min_{q\in\Delta^K}
\left(\langle q, a\rangle+\frac{\gamma}{2}(\|q\|_2^2-1)\right).$$

Its gradient is the minimizing vector $q$, which can contain exact zeros.
It is differentiable, but can exclude sufficiently expensive components
completely.

Smoothmin is the softargmin-weighted average of the inputs:

$$\mu^\gamma_{\rm smooth}(a)=\sum_{k=1}^K q_k^\gamma a_k,\qquad
\frac{\partial\mu^\gamma_{\rm smooth}}{\partial a_k}
=q_k^\gamma\left(1-\frac{a_k-\mu^\gamma_{\rm smooth}}{\gamma}\right).\qquad \mathrm{with}\quad q_k^\gamma=\frac{\partial\mu^\gamma_{\rm soft}}{\partial a_k}$$

It bounds the minimum function from above and permits negative gradients. Compare this behavior with the nonnegative softmin gradients.

In [ ]:
min_functions = [("Sparsemin", sparsemin), ("Smoothmin", smoothmin)]
gammas = [1e-10, 0.5, 1.0, 2.0]

a1 = np.linspace(0, 4, 100)
a2 = 2

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(6.2, 4.2))
for i_func, (name, min_func) in enumerate(min_functions):
    for i_gamma, gamma in enumerate(gammas):
        vals = []
        grads = []
        for a1_ in a1:
            val, grad = min_func(torch.tensor([a1_, a2]), gamma=gamma)
            vals.append(val.item())
            grads.append(grad[0].item())
        ax[0, i_func].plot(a1, vals, label="gamma")
        ax[0, i_func].set_xticks([0, 1, 2, 3, 4])
        ax[0, i_func].set_title(name)

        ax[1, i_func].plot(a1, grads, label="gamma")
        ax[1, i_func].set_xlabel("a1")
        ax[1, i_func].set_xticks([0, 1, 2, 3, 4])
        # val, grad = min_func(torch.tensor([a1, a2]), gamma=gamma, dim=0)
ax[0, 0].set_ylabel("$\mu(a_1, 2)$")
ax[1, 0].set_ylabel("$\partial \mu(a_1, 2)/\partial a_1$")

ax[0, 0].legend(
    [f"$\gamma={gamma}$" for gamma in gammas], loc="lower right", fontsize=8
)

for a in ax.flatten():
    a.grid()
    a.set_xlim([0, 4])

fig.tight_layout()
plt.show()

In [ ]:
a1 = np.arange(0, 4.25, 0.25)
a2 = np.arange(0, 4.25, 0.25)
gammas = [1e-10, 0.5, 2.0]

cmap_rbg = LinearSegmentedColormap.from_list(
    "BlueBlackRed", ["tab:blue", "black", "tab:red"]
)

In [ ]:
for name, min_func in min_functions:
    fig, ax = plt.subplots(2, 3, figsize=(6.2, 4.2))

    for i, gamma in enumerate(gammas):
        a1_, a2_ = np.meshgrid(a1, a2)

        U = np.zeros_like(a1_)
        V = np.zeros_like(a2_)

        S = np.zeros_like(a1_)

        for n in range(a1_.shape[0]):
            for m in range(a1_.shape[1]):
                val, grads = min_func(
                    torch.tensor([a1_[n, m], a2_[n, m]]), gamma=gamma
                )

                S[n, m] = val
                U[n, m] = grads[0]
                V[n, m] = grads[1]

        # ax[0,i].contour(a1_, a2_, S, cmap="viridis", levels=20)
        ax[0, i].remove()
        ax[0, i] = fig.add_subplot(2, 3, i + 1, projection="3d")
        ax[0, i].plot_surface(
            a1_,
            a2_,
            S,
            cmap="hot_r",
            edgecolor="black",
            linewidth=0.12,
            antialiased=True,
        )
        ax[0, i].set(
            xlim=[0, 4],
            ylim=[0, 4],
            xticks=[0, 1, 2, 3, 4],
            yticks=[0, 1, 2, 3, 4],
            box_aspect=(1, 1, 0.75),
        )
        ax[0, i].view_init(elev=15, azim=-115)
        
        ax[0, i].set_xlabel(r"$a_1$", labelpad=-4)
        ax[0, i].set_ylabel(r"$a_2$", labelpad=-4)
        ax[0, i].zaxis.set_rotate_label(False)  # disable automatic rotation
        ax[0, i].set_zlabel(r"$\mu(a_1,a_2)$", rotation=90, labelpad=-4)#, labelpad=2)

        # Move tick labels closer to the grid
        ax[0, i].tick_params(axis="x", pad=-2)
        ax[0, i].tick_params(axis="y", pad=-2)
        ax[0, i].tick_params(axis="z", pad=0)

        ax[1, i].quiver(
            a1, a2, U, V, U - V, cmap=cmap_rbg, headwidth=3, width=0.01
        )
        ax[1, i].set_xlabel("a1")

        ax[1, i].set_xlim([0, 4])
        ax[1, i].set_ylim([0, 4])
        ax[0, i].set_title("gamma = %.1f" % gamma)

    ax[0, 0].set_ylabel("a2")
    ax[1, 0].set_ylabel("a2")
    fig.suptitle(name, fontsize=9)
    fig.tight_layout()
    plt.show()

<a id="why_softmin"></a>

<div style="
  background-color: #C8E6C9A0;
  border: 1px solid #81C784A0;
  padding: 10px 20px;
  margin: 10px 0;
">

<h2 style="color:#1B5E20; margin-top:0;">Background: Why SDTW Uses Softmin</h2>

<p style="color:#1B5E20;">The equivalence of the DP recursions of SDTW relies on the associativity
and distributivity of the differentiable minimum function. Softmin satisfies these properties, therefore
nested soft minima in the dynamic-programming recurrence reproduce one
global soft minimum over all path costs. The SDTW gradient has the interpretation
of a Gibbs-weighted expected alignment.</p>
<p style="color:#1B5E20;">Sparsemin and smoothmin do not in general satisfy the same associativity
property. Substituting them into the local recurrence gives alternative
differentiable alignment objectives, but the DP solution is not equivalent to the global problem. 
Smoothmin's possibly negative derivatives also prevent the same probabilistic interpretation.</p>

</div>

<a id="exercises"></a>

## Exercises and Solutions


The soft minimum satisfies

$$\min(a)-\gamma\log K\leq\mu^\gamma(a)\leq\min(a),$$

$$\mu^\gamma(a+c\mathbf{1})=\mu^\gamma(a)+c,$$

$$\mu^\gamma(a,\mu^\gamma(b,c))=\mu^\gamma(a,b,c).$$

<a id="exercise_properties"></a>

<div style="
  background-color: #B2EBF290;
  border: 1px solid #4DD0E190;
  padding-top: 10px;
  padding-bottom: 10px;
  padding-left: 20px;
  padding-right: 20px;
  margin-top: 10px;
  margin-bottom: 10px;
">

<strong style="color:#006064; display:block; break-after:avoid; page-break-after:avoid;">Exercise 1: Properties of the Soft Minimum</strong>

<ol style="color:#006064;">
<li>Relate the 1D plots to the surfaces and gradient fields.</li>
<li>Derive the upper and lower bounds. When is either bound attained?</li>
<li>Prove the shift and associativity properties above.</li>
<li>Explain why adding a positive-infinity entry leaves softmin unchanged, and verify this numerically.</li>
</ol>

</div>

### Solution: Properties of the Soft Minimum

1. The value plot is the slice $a_2=2$ through the surface. Its derivative
   is the first component of the vector field along that slice. At a tie
   the softmin gradient is $(1/2,1/2)$; away from it, smaller temperatures
   concentrate mass on the smaller input.
2. Let $m=\min_k a_k$ and $S=\sum_{k=1}^K e^{-(a_k-m)/\gamma}$. Since
   $1\leq S\leq K$, $\mu^\gamma=m-\gamma\log S$ gives both bounds.
   The lower bound is attained when all entries are equal. For $K>1$ and
   finite entries the upper bound is strict; it is approached when all
   nonminimal entries become infinitely expensive. It is attained for
   $K=1$, or with just one finite entry and all others $+\infty$.
3. Factoring $e^{-c/\gamma}$ out of the sum proves the shift property.
   For associativity, set
   $e^{-\mu^\gamma(b,c)/\gamma}=e^{-b/\gamma}+e^{-c/\gamma}$.
   Substitution gives $-\gamma\log(e^{-a/\gamma}+e^{-b/\gamma}
   +e^{-c/\gamma})$ for both sides. 
4. An infinite entry adds $e^{-\infty/\gamma}=0$ to the sum. Its gradient
   component is zero, so it contributes neither cost nor gradient mass.

In [ ]:
finite = torch.tensor([1.0, 2.0], dtype=torch.float64)
with_infinity = torch.cat([finite, torch.tensor([torch.inf])])
value, _ = softmin(finite, gamma=0.5)
extended_value, extended_grad = softmin(with_infinity, gamma=0.5)
torch.testing.assert_close(value, extended_value)
assert extended_grad[-1] == 0
print("Finite and extended softmin:", value.item(), extended_value.item())

<a id="exercise_stability"></a>

<div style="
  background-color: #B2EBF290;
  border: 1px solid #4DD0E190;
  padding-top: 10px;
  padding-bottom: 10px;
  padding-left: 20px;
  padding-right: 20px;
  margin-top: 10px;
  margin-bottom: 10px;
">

<strong style="color:#006064; display:block; break-after:avoid; page-break-after:avoid;">Exercise 2: Numerical Stability</strong>

<ol style="color:#006064;">
<li>Implement softmin directly for two inputs. Use gamma = 1 and a1 = 0; vary a2 from −10,000 to zero.</li>
<li>Identify the operation that overflows for large negative inputs.</li>
<li>Use shift equivariance to implement a stable version and compare.</li>
</ol>

</div>

### Solution: Numerical Stability

Direct evaluation overflows in $\exp(-a_2/\gamma)$ for large negative
$a_2$, giving a spurious $-\infty$. Subtract $m=\min(a_1,a_2)$ inside
the exponential and add it back outside:

$$\mu^\gamma(a_1,a_2)=m-\gamma\log
(e^{-(a_1-m)/\gamma}+e^{-(a_2-m)/\gamma}).$$

The exponential arguments are nonpositive and one exponential is one.

In [ ]:
def softmin_naive(x1, x2, gamma):
    return -gamma * np.log(np.exp(-x1 / gamma) + np.exp(-x2 / gamma))


def softmin_stable(x1, x2, gamma):
    x_min = np.minimum(x1, x2)
    return x_min - gamma * np.log(
        np.exp(-(x1 - x_min) / gamma) + np.exp(-(x2 - x_min) / gamma)
    )

lowest_val = -1e3

x2 = np.linspace(lowest_val, 10, 1001)
with np.errstate(over="ignore"):
    naive_values = softmin_naive(0, x2, gamma=1)
stable_values = softmin_stable(0, x2, gamma=1)
reference = -np.logaddexp(0, -x2)
np.testing.assert_allclose(stable_values, reference)
print("Nonfinite naive values:", np.count_nonzero(~np.isfinite(naive_values)))

fig, ax = plt.subplots(1, 2, figsize=(6.2, 2.2), layout="tight")
for a in ax:
    a.plot(x2, naive_values, label="naive", linewidth=4, color="tab:red", alpha=.5)
    a.plot(x2, stable_values, "--", label="stable")
    a.grid()
ax[0].legend()
ax[1].set_xlim([-5, 5])
ax[1].set_ylim([-5, 5])
ax[0].set_xlabel("$x_2$")
ax[0].set_ylabel("Softmin")
ax[1].set_xlabel("$x_2$")
plt.show()